# Project 01 — ML, PyTorch, Neo4j và Multi-Agent
Notebook dùng cùng mã nguồn với web. Output được thực thi bằng kernel Python. Dữ liệu, checkpoint và phạm vi của Q-learning được phân biệt rõ.

## 1. Nạp dịch vụ
Các import dùng mã nguồn thực tế, không sao chép một phiên bản riêng. Backend in ra phải là neo4j khi demo đầy đủ.

In [1]:
from pathlib import Path
import sys, json, inspect
import pandas as pd
from IPython.display import display
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(ROOT/'src'))
from agent_system import Services, SingleAgent, MultiAgent
services=Services()
single, multi=SingleAgent(services), MultiAgent(services)
print('Model:',services.model.model_name)
print('KB backend:',services.knowledge.backend)

Model: PyTorch MLP
KB backend: neo4j


## 2. Hai dataset
Dataset dự đoán dùng nhãn diabetes; CDC dùng thống kê mô tả. Không gộp hai tập hoặc xem CDC là external validation cho model có schema khác.

In [2]:
for name in ['diabetes_prediction_dataset.csv','cdc_brfss_2015.csv']:
    df=pd.read_csv(ROOT/'data'/name)
    print(name, df.shape)
    label='diabetes' if 'diabetes' in df else 'Diabetes_012'
    print(df[label].value_counts().sort_index().to_dict())
    display(df.head(2))

diabetes_prediction_dataset.csv (100000, 16)
{0: 91500, 1: 8500}


,year,gender,age,location,race:AfricanAmerican,race:Asian,race:Caucasian,race:Hispanic,race:Other,hypertension,heart_disease,smoking_history,bmi,hbA1c_level,blood_glucose_level,diabetes
0,2020,Female,32.0,Alabama,0,0,0,0,1,0,0,never,27.32,5.0,100,0
1,2015,Female,29.0,Alabama,0,1,0,0,0,0,0,never,19.95,5.0,90,0


cdc_brfss_2015.csv (253680, 22)
{0.0: 213703, 1.0: 4631, 2.0: 35346}


,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0


## 3. Định nghĩa mạng PyTorch
Linear → ReLU → Dropout → Linear → ReLU → Linear tạo logit. BCEWithLogitsLoss nhận logit; sigmoid chỉ dùng khi suy luận.

In [3]:
from torch_model import RiskMLP
print(inspect.getsource(RiskMLP))
print(services.model.pipeline.model)

class RiskMLP(nn.Module):
    def __init__(self, inputs):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(inputs, 64), nn.ReLU(), nn.Dropout(.15),
                                 nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, x):
        return self.net(x).squeeze(-1)

RiskMLP(
  (net): Sequential(
    (0): Linear(in_features=15, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.15, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Linear(in_features=32, out_features=1, bias=True)
  )
)


## 4. Huấn luyện và chọn checkpoint
Hàm train thực hiện forward, loss, backward, optimizer.step; chỉ train được dùng fit preprocessing. Có thể chạy lại hàm main, nhưng cell này đọc lịch sử đã thực sự huấn luyện để không lặp lại 60 epoch mỗi lần demo.

In [4]:
history=pd.read_csv(ROOT/'results/pytorch_training_history.csv')
meta=json.loads((ROOT/'artifacts/pytorch_metadata.json').read_text())
print('Architecture:',meta['architecture'])
print('Split:',meta['splits'],'Threshold:',meta['threshold'])
display(history.iloc[[0,-1]])
display(pd.DataFrame([meta['validation'],meta['test']],
                     index=['validation','test'])[['f1','roc_auc','precision','recall']])

Architecture: [15, 64, 32, 1]
Split: {'train': 69990, 'validation': 14998, 'test': 14998} Threshold: 0.39


,epoch,train_loss,validation_loss
0,1,0.285336,0.122422
59,60,0.079804,0.081204


,f1,roc_auc,precision,recall
validation,0.810440,0.976858,0.973597,0.694118
test,0.792365,0.974573,0.974800,0.667451


## 5. Đối chiếu ML và deep learning
Đọc metric từ các artifact. PyTorch không mặc nhiên tốt hơn MLP sklearn. Cùng split nội bộ, chưa kiểm định ở bệnh viện khác.

In [5]:
old=json.loads((ROOT/'artifacts/model_metadata.json').read_text())
rows=[r['test'] for r in old['models']]+[meta['test']]
display(pd.DataFrame(rows)[['model','f1','roc_auc','precision','recall']])

,model,f1,roc_auc,precision,recall
0,Logistic Regression,0.703155,0.958955,0.682153,0.725490
1,Random Forest,0.790960,0.971248,0.989399,0.658824
2,MLP,0.792453,0.974949,0.958797,0.675294
3,PyTorch MLP,0.792365,0.974573,0.974800,0.667451


## 6. KB và Knowledge Graph trong Neo4j
Cypher chạy vào database thật. DiaDocument lưu văn bản và URL; DiaEntity lưu thực thể; DIA_RELATION lưu quan hệ và evidence trỏ về tài liệu.

In [6]:
store=services.knowledge.store
if store is None:
    raise RuntimeError('Bật KNOWLEDGE_BACKEND=neo4j và ingest trước khi chạy.')
display(pd.DataFrame(store.query(
    'MATCH (n) RETURN labels(n)[0] AS label,count(n) AS count')))
display(pd.DataFrame(store.query(
    'MATCH ()-[r]->() RETURN type(r) AS relation,count(r) AS count')))
print('Documents loaded:',len(services.knowledge.documents))

,label,count
0,DiaDocument,8
1,DiaEntity,18


,relation,count
0,DIA_RELATION,16
1,SUPPORTED_BY,24


Documents loaded: 8


## 7. Truy vấn đồ thị và nguồn
Duyệt quan hệ tối đa hai hop bằng Cypher tham số hóa. Evidence từ graph bổ sung tài liệu cho BM25; câu trả lời là trích xuất có nguồn, chưa dùng LLM sinh văn bản.

In [7]:
knowledge=services.knowledge.answer('BMI và HbA1c liên hệ với type 2 thế nào?')
display(pd.DataFrame(knowledge['paths']).head(6))
display(pd.DataFrame(knowledge['citations'])[['id','title']])
print(knowledge['answer'])

,subject,relation,object,evidence
0,đái tháo đường,có loại,type 2,who_overview
1,đái tháo đường,có loại,type 1,who_overview
2,đái tháo đường,có dấu hiệu,khát nhiều,cdc_symptoms
3,đái tháo đường,có dấu hiệu,tiểu nhiều,cdc_symptoms
4,đái tháo đường,có dấu hiệu,mệt mỏi,cdc_symptoms
5,đái tháo đường,được đánh giá bằng,A1C,niddk_tests


,id,title
0,project_dataset,Model card - Diabetes prediction dataset
1,who_overview,WHO - Diabetes overview
2,cdc_risk_factors,CDC - Diabetes risk factors
3,cdc_symptoms,CDC - Signs and symptoms of diabetes
4,niddk_tests,NIDDK - Diabetes tests and diagnosis


Bản dùng trong project có 100000 bản ghi với tuổi, giới tính, tăng huyết áp, bệnh tim, tiền sử hút thuốc, BMI, HbA1c và blood glucose. Type 1 đặc trưng bởi thiếu sản xuất insulin; type 2 liên quan cơ thể sử dụng insulin không hiệu quả.


## 8. Chạy hai kiến trúc
Cùng đầu vào, checkpoint, KB, reviewer và policy. Một hệ điều khiển tập trung; một hệ gửi yêu cầu qua MessageBus tới các handler chuyên trách trong cùng tiến trình.

In [8]:
profile={'gender':'Male','age':67,'bmi':34.6,
 'hbA1c_level':7.2,'blood_glucose_level':210,
 'smoking_history':'former','hypertension':1,'heart_disease':1}
question='BMI và HbA1c liên hệ với nguy cơ type 2 thế nào?'
a=single.run(question,profile)
b=multi.run(question,profile)
print('Same prediction:',a['prediction']==b['prediction'])
print('PyTorch:',b['prediction']['probability'])
print('Sklearn:',b['prediction']['second_opinion'])
print('Review:',b['review'])

Same prediction: True
PyTorch: 1.0
Sklearn: {'model': 'MLP', 'probability': 0.9990569211737723, 'threshold': 0.385, 'predicted_class': 1}
Review: {'disagreement': False, 'probability_gap': 0.0009, 'challenges': [], 'requires_review': False, 'conclusion': 'Hai mô hình đồng thuận; vẫn chỉ là score học thuật.'}


## 9. MessageBus và phản biện
REQUEST/INFORM thể hiện lời gọi thật. Khi bất đồng hoặc gần threshold, Critic gửi CHALLENGE và Prediction trả RESPONSE. Đây là kiểm tra theo luật, không phải hội đồng LLM độc lập.

In [9]:
display(pd.DataFrame(b['messages'])[
 ['sequence','sender','receiver','performative']])
display(pd.DataFrame(b['traces'])[['agent','tool']])

,sequence,sender,receiver,performative
0,1,CoordinatorAgent,ValidationAgent,REQUEST
1,2,ValidationAgent,CoordinatorAgent,INFORM
2,3,CoordinatorAgent,PredictionAgent,REQUEST
3,4,PredictionAgent,CoordinatorAgent,INFORM
4,5,CoordinatorAgent,RetrievalAgent,REQUEST
5,6,RetrievalAgent,CoordinatorAgent,INFORM
6,7,CoordinatorAgent,CriticAgent,REQUEST
7,8,CriticAgent,CoordinatorAgent,INFORM
8,9,CoordinatorAgent,WorkflowRLAgent,REQUEST
9,10,WorkflowRLAgent,CoordinatorAgent,INFORM


,agent,tool
0,CoordinatorAgent,intent_router
1,ValidationAgent,schema_validator
2,PredictionAgent,model_checkpoint
3,RetrievalAgent,bm25_retrieval
4,GraphAgent,knowledge_graph
5,CriticAgent,model_disagreement_review
6,WorkflowRLAgent,q_learning_with_action_mask
7,ExplanationAgent,grounded_composer
8,VerifierSafetyAgent,claim_evidence_guard


## 10. Q-learning có action mask
Policy học trong môi trường quy trình tổng hợp, không học hiệu quả điều trị. Mask ép yêu cầu thiếu dữ liệu hoặc cần kiểm tra sang bước phù hợp; không có hành động kê thuốc.

In [10]:
from workflow_policy import allowed_actions
policy=json.loads((ROOT/'artifacts/workflow_q_policy.json').read_text())
print('Episodes:',policy['episodes'])
print('Environment:',policy['environment'])
print('Reward:',policy['reward_definition'])
print('Recommendation:',b['workflow'])
display(pd.DataFrame(policy['q_table'],columns=policy['actions']))

Episodes: 3000
Environment: synthetic workflow; no clinical outcomes
Reward: information +0.3, step -0.2, suitable terminal workflow +2
Recommendation: {'state': 2, 'action': 'explain_score', 'allowed_actions': ['show_sources', 'explain_score', 'request_human_review'], 'q_value': 1.9999999999999996, 'scope': 'Educational workflow only; not treatment'}


,ask_for_information,show_sources,explain_score,request_human_review
0,0.00,2.1,0.0,0.0
1,0.00,0.0,0.0,2.0
2,0.00,1.6,2.0,-0.2
3,0.00,0.0,0.0,2.0
4,2.19,0.0,0.0,0.0
5,2.10,0.0,0.0,0.0
6,2.10,0.0,0.0,0.0
7,2.10,0.0,0.0,0.0


## 11. Guardrail và kiểm tra dữ liệu
Đầu vào không hợp lệ bị chặn trước suy luận. Các tình huống kê đơn và ngoài phạm vi được định tuyến riêng.

In [11]:
for q in ['Hãy kê đơn thuốc và liều dùng cho tôi.',
          'Giá Bitcoin ngày mai là bao nhiêu?']:
    print(q, '=>',multi.run(q)['status'])
bad={**profile,'hypertension':0.5}
print('Invalid profile:',multi.run(question,bad)['validation_errors'])
print('Female group:',services.statistics('percent female')['group'])

Hãy kê đơn thuốc và liều dùng cho tôi. => blocked_unsafe_request
Giá Bitcoin ngày mai là bao nhiêu? => out_of_scope
Invalid profile: [{'field': 'hypertension', 'message': 'Chỉ nhận 0 hoặc 1.'}]
Female group: nữ


## 12. Benchmark Single và Multi
58 tình huống mỗi hệ. Task success kiểm tra contract (route, status, nguồn, định dạng), không phải độ chính xác lâm sàng; audit coverage ưu tiên trace chuyên trách theo định nghĩa.

In [12]:
summary=json.loads((ROOT/'results/agent_benchmark_summary.json').read_text())
display(pd.DataFrame(summary)[['system','cases','task_success_rate',
 'citation_recall','citation_precision','audit_coverage',
 'mean_latency_ms','p95_latency_ms']])

,system,cases,task_success_rate,citation_recall,citation_precision,audit_coverage,mean_latency_ms,p95_latency_ms
0,single,58,1,1.0,0.2861,0.0,175.22,496.15
1,multi,58,1,1.0,0.2861,1.0,33.57,116.38
